In [1]:
import pandas as pd

df_ground_truth=pd.read_csv('data/ground_truth-new.csv')
ground_truth=df_ground_truth.to_dict(orient='records')

In [2]:
ground_truth[10]

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [44]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
documents_llm=[]

for i in documents:
    if i["course"]=='llm-zoomcamp':
        documents_llm.append(i)

documents=documents_llm

index=build_index(documents)

In [52]:
documents[9]

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Are there any lectures/videos? Where are they?',
 'answer': 'Use the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp) as the main entry point.\n\nOpen the lesson folders in the repo. Each lesson page has the relevant videos linked at the top.\n\n<{IMAGE:lessons}>\n\nWhen in doubt, follow the GitHub repo first, because it is easier to keep updated than the YouTube playlist.',
 'doc_id': '31456f4b5f'}

In [45]:
doc_idx={}

for i in documents:
    doc_idx[i["doc_id"]]=i

In [61]:
len(doc_idx)

113

In [63]:
ground_truth_f = [q for q in ground_truth if q["document"] in doc_idx]

len(ground_truth_f)

315

In [5]:
q=ground_truth[10]
q

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [6]:
doc_idx[q["document"]]

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.',
 'doc_id': '489dd1c9d9'}

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
    course='llm-zoomcamp'
)

In [9]:
q['question']

'How do I join the Office Hours or live workshop if I don’t have the Zoom link?'

In [10]:
answer=assistant.rag(q['question'])

In [11]:
assistant.total_cost()

0.00122625

In [12]:
print(answer)

The Zoom link is only published to instructors/presenters/TAs.

If you're a student, join via:
- **YouTube Live** on the DataTalksClub YouTube channel
- **Slido** for questions (the link is pinned in chat when live)
- The session video URL is also posted in the **announcements channel on Telegram and Slack** before it starts

If you can’t find the link, check those announcements.


In [13]:
doc_id=q['document']
orig_doc=doc_idx[doc_id]
orig_ans=orig_doc["answer"]

print(orig_ans)

The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).

Don’t post questions in chat as they may be missed if the room is very active.


In [14]:
rag_result = {
    "question": q['question'],
    "answer_llm": answer,
    "answer_orig": orig_ans,
    "document": doc_id,
}

rag_result

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'answer_llm': "The Zoom link is only published to instructors/presenters/TAs.\n\nIf you're a student, join via:\n- **YouTube Live** on the DataTalksClub YouTube channel\n- **Slido** for questions (the link is pinned in chat when live)\n- The session video URL is also posted in the **announcements channel on Telegram and Slack** before it starts\n\nIf you can’t find the link, check those announcements.",
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very 

Processing all questions

In [15]:
def generate_rag_answer(q):
    question=q["question"]
    doc_id=q["document"]
    orig_doc=doc_idx[doc_id]

    answer_llm=assistant.rag(question)
    answer_orig=orig_doc["answer"]

    result={
        "question":question,
        "answer_llm":answer_llm,
        "answer_orig":answer_orig,
        "document":doc_id
    }

    return result

In [16]:
answer1= generate_rag_answer(ground_truth[0])

answer1

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, you can still join late. You can start learning now, and if you want a certificate, make sure to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [31]:
test=generate_rag_answer(ground_truth[15])

test

KeyError: 'c6c2888275'

In [40]:
ground_truth[:20]

[{'question': 'Is it okay to join the course late if I just found it now?',
  'document': '74eb249bbf'},
 {'question': 'Can I still take this course even if I missed the start date?',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has already started, am I still eligible for a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to submit my project before submissions close to get the certificate?',
  'document': '74eb249bbf'},
 {'question': 'I’m a bit late to the course—what do I need to do to still earn the certificate?',
  'document': '74eb249bbf'},
 {'question': 'I registered for the LLM Zoomcamp — when should I expect a confirmation email?',
  'document': '977bf7786c'},
 {'question': 'Do I actually need an acceptance email before I can start the course and hand in homework?',
  'document': '977bf7786c'},
 {'question': 'If I filled out the registration form, does that mean I’m officially on a checked list for the course?',
  'document': '977b

In [43]:
len(doc_idx)

113

In [67]:
assistant.total_cost()

0.0052545

In [68]:
assistant.reset_usage()

In [69]:
assistant.total_cost()

0.0

In [66]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [70]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth_f, generate_rag_answer)

  0%|          | 0/315 [00:00<?, ?it/s]

In [71]:
assistant.total_cost()

0.3118634999999999

In [72]:
answers=[]

for i in results:
    answers.append(i)

In [73]:
df_answers=pd.DataFrame(answers)
df_answers

,question,answer_llm,answer_orig,document
0,Is it okay to join the course late if I just f...,"Yes, you can still join the course late. If yo...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,Can I still take this course even if I missed ...,"Yes. You can start whenever you want, even if ...","Yes, but if you want to receive a certificate,...",74eb249bbf
2,If I join after the course has already started...,Yes — you can still join after the course has ...,"Yes, but if you want to receive a certificate,...",74eb249bbf
3,Do I need to submit my project before submissi...,"Yes. To get the certificate, you need to submi...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,I’m a bit late to the course—what do I need to...,"Yes — you can still join, but to earn the cert...","Yes, but if you want to receive a certificate,...",74eb249bbf
...,...,...,...,...
310,Why do I get a 401 Client Error when using the...,A 401 Client Error usually means an authorizat...,"If you encounter a 401 Client Error, it may in...",4b30b918bc
311,What's the easiest way to force-install reques...,Use this pip command to install the correct Re...,"If you encounter a 401 Client Error, it may in...",4b30b918bc
312,Can I install requests straight from the GitHu...,"Yes. Use:\n\n```bash\npip install ""requests @ ...","If you encounter a 401 Client Error, it may in...",4b30b918bc
313,"If pip keeps pulling requests v2.28, what exac...","Run:\n\n```bash\npip install ""requests @ https...","If you encounter a 401 Client Error, it may in...",4b30b918bc


In [74]:
df_answers.to_csv('data/rag-answers-new.csv', index=False)